# qibocal-widgets in Jupyter

A tour of the features of `qibocal-widgets`, on a simulated platform: the `qutrits` platform of [qibolab-simulator](https://github.com/qiboteam/qibolab-simulator), copied in the `platforms/` folder of the examples.

The executor is a drop-in replacement of Qibocal's, composed of **frontends** (displaying every output), a **runner** (where the steps run), and **middleware** (wrapping every step).

In [ ]:
import json
import os
import shutil
import signal
import time
from pathlib import Path

import ipywidgets as widgets
from IPython.display import Markdown
from qibocal import Executor as QibocalExecutor
from qibocal import create_calibration_platform, protocols

from qibocal_widgets import (
    Executor,
    Middleware,
    PlatformState,
    data_frames,
    reports,
    reports_dir,
)
from qibocal_widgets.jupyter import Jupyter, display, show_data, show_reports, wrap

HERE = Path.cwd()  # Jupyter runs notebooks in their folder
os.environ["QIBOLAB_PLATFORMS"] = str(HERE.parent / "platforms")


def qutrits():
    """A fresh simulated platform: two coupled qutrits.

    qibolab replaces the interrupt and termination handlers: they are restored
    right after creating the platform, so that interrupting a cell behaves as
    usual.
    """
    handlers = {s: signal.getsignal(s) for s in (signal.SIGINT, signal.SIGTERM)}
    platform = create_calibration_platform("qutrits")
    for number, handler in handlers.items():
        signal.signal(number, handler)
    return platform

## 1. The executor and its session

Create the executor once, at the beginning of the notebook. Every execution is recorded in its *session* folder, in Qibocal's output layout. Within a session started by the Qibocal application (or `qibocal notebook`), the folder defaults to a fresh subfolder of the session's reports folder (`QIBOCAMP_REPORTS_DIR`), so that calibrations show up in its dashboard; otherwise, it is given explicitly.

In [ ]:
if reports_dir() is None:
    path = HERE / "var" / "session"
    if path.exists():
        shutil.rmtree(path)
else:
    path = None  # a fresh folder under reports_dir()

executor = Executor.create(
    path=path,
    platform=qutrits(),
    targets=[0, 1],
    update=False,
    frontends=[Jupyter()],
)
executor.path

## 2. Implicit display, step by step

Every step returns an execution which displays itself: raw data after the acquisition, then reports (selected by default), with the fit table folded above the plots.

In [ ]:
bound = protocols.ramsey(delay=(10, 1000, 100))
with executor:
    acquired = executor.acquire(bound, nshots=500)
acquired

In [ ]:
fitted = executor.fit(acquired)
ramsey = executor.report(fitted)
ramsey

The execution is recorded in the session, where notes can be attached to it (and to the session itself): they are shown, and can be added, in the report dashboard too.

In [ ]:
executor.note("Two coupled qutrits, simulated")
executor.note("Fringes as expected", ramsey)
{
    "task folder": ramsey.path.relative_to(executor.path),
    "notes": [note.content for note in executor.session.notes(ramsey)],
}

## 3. Updating the platform

A full execution (`executor(bound)`) acquires, fits, and reports. The executor is created with `update=False`: the platform is updated once the fit is validated, with the *Update* buttons above the reports (all targets, or per target, inside each report), or from code with `rabi.update()`. (With `update=True`, it is updated right away, and the buttons show it.) A snapshot of the platform state taken before shows what changed.

In [ ]:
start = executor.platform_state.snapshot()
with executor:
    rabi = executor(
        protocols.rabi_amplitude_signal(
            min_amp=0.0, max_amp=0.6, step_amp=0.03, pulse_length=40
        ),
        nshots=500,
    )
rabi

In [ ]:
rabi.update([0])  # the fit of target 0 looks right: approve it alone
executor.platform_state.diff(start)

## 4. Exploring and editing the platform state

The explorer shows parameters and calibration by path, highlighting the differences with the snapshot. Its navigation bar accepts paths, or pasted code; buttons copy paths and values; with `editable=True`, edits go through the same API, and show the equivalent code.

In [ ]:
executor.platform_state.explore(
    "native_gates.single_qubit.0", editable=True, compare=start
)

The same paths work from code, in any of the forms the explorer accepts: full paths, paths within a document (as copied from the dashboard), or pasted code.

In [ ]:
state = executor.platform_state
frequency = "parameters.configs.0/drive.frequency"
before = state[frequency]
state.update({"configs.0/drive.frequency": before + 1e6})
{
    "before": before,
    "after": state[frequency],
    "resolved": state.resolve(f'executor.platform_state["{frequency}"]'),
    "qubit frequencies": state.find("frequency_01"),
    "session start": PlatformState.load(executor.path / "platform")[frequency],
}

## 5. Explicit helpers

For custom layouts, the views are available explicitly, as well as the frontend-agnostic content (tables and normalized reports). Executions from a plain Qibocal executor are made displayable with `wrap`.

In [ ]:
tabs = widgets.Tab(children=[show_data(ramsey), show_reports(ramsey), display(ramsey)])
for index, title in enumerate(["show_data", "show_reports", "display"]):
    tabs.set_title(index, title)
tabs

In [ ]:
frames = data_frames(ramsey.data)
normalized = reports(ramsey)
Markdown(
    f"`data_frames`: {len(frames)} tables, columns {list(frames[0].columns)}\n\n"
    f"`reports`: {len(normalized[0].figures)} figure(s) per target, "
    f"fit table of {len(normalized[0].table)} characters"
)

In [ ]:
platform = qutrits()
plain = QibocalExecutor(platform, targets=[0], update=False)
platform.connect()
completed = plain(protocols.ramsey(delay=(10, 1000, 100)), nshots=200)
platform.disconnect()
wrap(completed)

## 6. Middleware

Middleware wrap every step, like in web servers. A timer, around the acquisition, on a second executor appending to the same session folder (`Executor.attach`, e.g. after restarting the kernel): it restores the latest recorded platform state, so the calibration continues from there.

In [ ]:
class Timer(Middleware):
    def __init__(self):
        self.times = []

    def acquire(self, executor, call_next, *args, **kwargs):
        start = time.perf_counter()
        completed = call_next(*args, **kwargs)
        self.times.append(time.perf_counter() - start)
        return completed


timer = Timer()
attached = Executor.attach(
    executor.path,
    platform=qutrits(),
    targets=[0, 1],
    update=False,
    middleware=[timer],
    frontends=[Jupyter()],
)
with attached:
    attached(protocols.ramsey(delay=(10, 1000, 100)), nshots=500)
{
    "acquisition time": f"{timer.times[0]:.1f} s",
    "restored frequency": attached.platform_state["configs.0/drive.frequency"],
    "history": json.loads((attached.path / "history.json").read_text()),
}

## 7. Sessions on disk

Recorded executions can be loaded back, displayed, and the executor can move on to a new session folder, for instance for a new cooldown.

In [ ]:
history = json.loads((attached.path / "history.json").read_text())
attached.load(attached.path / "data" / history[0])

In [ ]:
first = attached.path
attached.new_session(first.parent / "next")
{"previous session": first.name, "new session": attached.path.name}

## Next: remote execution

To run the same protocols on a machine with access to the instruments, from a laptop or an access node, see the `remote` notebook next to this one.